# Feature matrix — basic stats (MVP)

Quick look at the Parquet written by `kernel-playground scan`.

```bash
cd explore
uv sync
uv run jupyter notebook explore_features.ipynb
# or: uv run explore-features
```

Defaults to `../features_20k.parquet`, else `/tmp/features_mainnet200.parquet`.


In [ ]:
from pathlib import Path

import polars as pl
import matplotlib.pyplot as plt

REPO_ROOT = Path("..").resolve()
CANDIDATES = [
    REPO_ROOT / "features_20k.parquet",
    Path("/tmp/features_mainnet200.parquet"),
]


def parquet_ready(path: Path) -> bool:
    if not path.exists() or path.stat().st_size == 0:
        return False
    try:
        pl.scan_parquet(path).collect_schema()
        return True
    except Exception as e:
        print(f"skip {path.name}: not readable yet ({type(e).__name__})")
        return False


PARQUET = next((p for p in CANDIDATES if parquet_ready(p)), None)
assert PARQUET is not None, "no readable parquet yet — wait for the scan to finish"

lf = pl.scan_parquet(PARQUET)
schema = lf.collect_schema()
print(f"file:   {PARQUET.resolve()}")
print(f"size:   {PARQUET.stat().st_size / 1e6:.1f} MB")
print(f"cols:   {len(schema)}")


## Overview

In [ ]:
overview = lf.select(
    pl.len().alias("n_txs"),
    pl.col("block_height").min().alias("height_min"),
    pl.col("block_height").max().alias("height_max"),
    (pl.col("block_height").max() - pl.col("block_height").min() + 1).alias("n_heights_span"),
    pl.col("block_height").n_unique().alias("n_heights_seen"),
    pl.col("is_coinbase").mean().alias("coinbase_rate"),
    pl.col("version").mean().alias("version_mean"),
).collect()

overview

## Tx version distribution

In [ ]:
n = overview["n_txs"][0]
versions = (
    lf.group_by("version")
    .agg(pl.len().alias("count"))
    .with_columns((pl.col("count") / n).alias("rate"))
    .sort("version")
    .collect()
)
versions

## Txs per block

In [ ]:
per_block = (
    lf.group_by("block_height")
    .agg(pl.len().alias("n_txs"))
    .sort("block_height")
    .collect()
)

stats = per_block.select(
    pl.col("n_txs").mean().alias("mean"),
    pl.col("n_txs").median().alias("median"),
    pl.col("n_txs").min().alias("min"),
    pl.col("n_txs").max().alias("max"),
    pl.col("n_txs").quantile(0.05).alias("p05"),
    pl.col("n_txs").quantile(0.95).alias("p95"),
)
display(stats)

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(per_block["block_height"], per_block["n_txs"], lw=0.6)
ax.set_xlabel("block height")
ax.set_ylabel("txs")
ax.set_title("transactions per block")
fig.tight_layout()
plt.show()

## Boolean feature prevalence

Fraction of txs where each fingerprint / heuristic column is true. Top and bottom by rate.

In [ ]:
bool_cols = [
    name for name, dtype in schema.items() if dtype == pl.Boolean and name != "is_coinbase"
]

prevalence = (
    lf.select([pl.col(c).mean().alias(c) for c in bool_cols])
    .collect()
    .transpose(include_header=True, header_name="feature", column_names=["rate"])
    .sort("rate", descending=True)
)

print(f"{len(bool_cols)} boolean feature columns")
print("\nTop 20 (most common):")
display(prevalence.head(20))
print("Bottom 20 (rarest):")
display(prevalence.tail(20))

In [ ]:
TOP_N = 25
top = prevalence.head(TOP_N)

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top["feature"].to_list()[::-1], top["rate"].to_list()[::-1])
ax.set_xlabel("fraction of txs")
ax.set_title(f"top {TOP_N} feature prevalence")
fig.tight_layout()
plt.show()

## Prevalence by feature family

In [ ]:
def family(name: str) -> str:
    for prefix in (
        "fp_",
        "h_",
        "prevout_type__",
        "output_type__",
        "change_heuristic__",
        "change_",
        "reveals_",
    ):
        if name.startswith(prefix):
            return prefix.rstrip("_")
    return "other"

by_family = (
    prevalence.with_columns(
        pl.col("feature").map_elements(family, return_dtype=pl.String).alias("family")
    )
    .group_by("family")
    .agg(
        pl.len().alias("n_features"),
        pl.col("rate").mean().alias("mean_rate"),
        pl.col("rate").max().alias("max_rate"),
        pl.col("rate").min().alias("min_rate"),
    )
    .sort("mean_rate", descending=True)
)
by_family

## Non-coinbase only

Same prevalence view with coinbases dropped (usually what you want for wallet fingerprints).

In [ ]:
lf_nc = lf.filter(~pl.col("is_coinbase"))
n_nc = lf_nc.select(pl.len()).collect().item()
print(f"non-coinbase txs: {n_nc:,} / {n:,} ({n_nc / n:.4%})")

prevalence_nc = (
    lf_nc.select([pl.col(c).mean().alias(c) for c in bool_cols])
    .collect()
    .transpose(include_header=True, header_name="feature", column_names=["rate"])
    .sort("rate", descending=True)
)
prevalence_nc.head(15)